In [4]:
import boto3
import time
import json
bedrock = boto3.client('bedrock', region_name='us-east-2')
response = bedrock.list_foundation_models()
print(f"Available models: {len(response['modelSummaries'])}")
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-2')

Available models: 96


In [5]:
import os
from botocore.exceptions import ClientError, BotoCoreError, EventStreamError

try:
    from IPython.display import display, Pretty
    _IN_NOTEBOOK = get_ipython().__class__.__name__ == 'ZMQInteractiveShell'
except Exception:
    _IN_NOTEBOOK = False

_RETRYABLE = {'ThrottlingException', 'ServiceUnavailableException', 'ModelNotReadyException'}


class _Status:
    """Single status line that is updated in place (notebook only)."""
    def __init__(self, enabled):
        self.handle = display(Pretty(""), display_id=True) if enabled and _IN_NOTEBOOK else None

    def update(self, line):
        if self.handle:
            self.handle.update(Pretty(line))


def _progress_bar(done, total, width=30):
    frac = min(done / total, 1.0) if total else 0
    filled = int(width * frac)
    return f"[{'█' * filled}{'░' * (width - filled)}] {frac:4.0%}"


def invoke_claude_stream(prompt, model_id='us.anthropic.claude-sonnet-4-5-20250929-v1:0',
                         max_tokens=500, temperature=None, top_k=None, top_p=None,
                         output_file=None, show_progress=True, max_retries=3):
    """Stream a response from Claude, printing text as it arrives.

    Shows a live progress bar + token counter, writes chunks to `output_file` as they
    arrive (if given) and never raises on AWS/stream errors: whatever was received is
    returned together with an `error` field.
    """
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }
    for key, value in (("temperature", temperature), ("top_k", top_k), ("top_p", top_p)):
        if value is not None:
            body[key] = value

    result = {"text": "", "input_tokens": None, "output_tokens": 0, "stop_reason": None,
              "completed": False, "error": None, "time_to_first_token": None, "elapsed": None}
    start = time.time()

    # 1. Open the stream (retry throttling with exponential backoff)
    response = None
    for attempt in range(max_retries + 1):
        try:
            response = bedrock_runtime.invoke_model_with_response_stream(
                modelId=model_id, body=json.dumps(body))
            break
        except ClientError as e:
            code, message = e.response['Error']['Code'], e.response['Error']['Message']
            if code in _RETRYABLE and attempt < max_retries:
                wait = 2 ** attempt
                print(f"{code}: retrying in {wait}s ({attempt + 1}/{max_retries})")
                time.sleep(wait)
                continue
            result["error"] = f"{code}: {message}"
            if code == 'ExpiredTokenException':
                result["error"] += " (refresh your AWS credentials)"
            break
        except BotoCoreError as e:  # e.g. no network, read timeout
            result["error"] = f"{type(e).__name__}: {e}"
            break
    if response is None:
        print(f"Stream failed to start - {result['error']}")
        return result

    # 2. Consume the stream
    out = open(output_file, "w", encoding="utf-8") if output_file else None
    status = _Status(show_progress)
    chunks = 0  # one text delta ~ one token; replaced by the exact count at the end
    try:
        for event in response["body"]:
            if "chunk" not in event:  # modeled error delivered mid-stream
                kind = next(iter(event))
                result["error"] = f"{kind}: {event[kind].get('message', '')}"
                break
            data = json.loads(event["chunk"]["bytes"])
            event_type = data.get("type")

            if event_type == "message_start":
                result["input_tokens"] = data["message"]["usage"]["input_tokens"]
            elif event_type == "content_block_delta" and data["delta"].get("type") == "text_delta":
                text = data["delta"]["text"]
                if result["time_to_first_token"] is None:
                    result["time_to_first_token"] = time.time() - start
                result["text"] += text
                chunks += 1
                print(text, end="", flush=True)
                if out:
                    out.write(text)
                    out.flush()  # partial output survives a crash
                elapsed = time.time() - start
                status.update(f"{_progress_bar(chunks, max_tokens)}  ~{chunks}/{max_tokens} tokens"
                              f"  |  {chunks / elapsed:.1f} tok/s  |  {elapsed:.1f}s")
            elif event_type == "message_delta":
                result["stop_reason"] = data["delta"].get("stop_reason")
                result["output_tokens"] = data["usage"]["output_tokens"]
            elif event_type == "message_stop":
                result["completed"] = True
    except KeyboardInterrupt:
        result["error"] = "Interrupted by user"
    except (EventStreamError, ClientError, BotoCoreError) as e:
        result["error"] = f"{type(e).__name__}: {e}"
    finally:
        response["body"].close()
        if out:
            out.close()

    result["elapsed"] = time.time() - start
    tokens = result["output_tokens"] or chunks
    status.update(f"{_progress_bar(tokens, max_tokens)}  {tokens}/{max_tokens} tokens"
                  f"  |  {result['elapsed']:.1f}s  |  stop: {result['stop_reason']}")
    print()
    if result["error"]:
        print(f"[Stream interrupted after ~{chunks} tokens - {result['error']}. Partial text kept.]")
    return result


def save_stream_to_file(prompt, path, **kwargs):
    """Stream a response and write it to `path` chunk by chunk."""
    result = invoke_claude_stream(prompt, output_file=path, **kwargs)
    if os.path.exists(path):
        state = "complete" if result["completed"] else "PARTIAL"
        print(f"Saved {state} response to {os.path.abspath(path)}")
    return result

In [6]:
# Live tokens counter + progress bar
result = invoke_claude_stream("Explain what AWS Bedrock is in 3 paragraphs", max_tokens=500)
if result["completed"]:
    print(f"\nInput tokens: {result['input_tokens']} | Output tokens: {result['output_tokens']} | "
          f"Time to first token: {result['time_to_first_token']:.2f}s | Total: {result['elapsed']:.2f}s")

# Save the stream to a file (written chunk by chunk)
save_stream_to_file("Give me an overview of the Mexican culture in 3 paragraphs",
                    "mexican_culture.txt", max_tokens=500)

# Error handling: a bad model id is reported instead of raising
invoke_claude_stream("Hello", model_id="us.anthropic.not-a-real-model")

[██████████████████░░░░░░░░░░░░]  61%  303/500 tokens  |  8.6s  |  stop: end_turn

# AWS Bedrock

**AWS Bedrock** is a fully managed service from Amazon Web Services that provides access to high-performing foundation models (FMs) from leading AI companies through a single API. Launched in 2023, it offers models from providers like Anthropic (Claude), AI21 Labs, Cohere, Meta (Llama), Stability AI, and Amazon's own Titan models. This eliminates the need for organizations to build relationships with multiple AI vendors or manage different integration points.

The service enables developers and businesses to build and scale generative AI applications without needing to manage the underlying infrastructure. Users can customize these foundation models with their own data through techniques like fine-tuning and Retrieval Augmented Generation (RAG), while keeping their data private and secure. Bedrock handles the heavy lifting of model deployment, scaling, and maintenance, allowing teams to focus on application development rather than ML operations.

A key advantage of AWS B

[████████████████████░░░░░░░░░░]  69%  347/500 tokens  |  8.3s  |  stop: end_turn

# Overview of Mexican Culture

Mexican culture is a vibrant blend of indigenous Mesoamerican civilizations and Spanish colonial influence, creating a unique identity that has evolved over thousands of years. The legacy of ancient cultures like the Aztec and Maya remains visible in language, cuisine, art, and traditions, while 300 years of Spanish rule introduced Catholicism, the Spanish language, and European customs. This mestizaje (mixing) forms the foundation of modern Mexican identity, characterized by strong family bonds, community ties, and a deep respect for both heritage and religious traditions.

Food is central to Mexican culture and has been recognized by UNESCO as an Intangible Cultural Heritage of Humanity. Mexican cuisine goes far beyond tacos and burritos, featuring complex dishes with regional variations that use indigenous ingredients like corn, beans, chili peppers, and chocolate. Celebrations and holidays are equally important, with Día de los Muertos (Day of the Dea

{'text': '',
 'input_tokens': None,
 'output_tokens': 0,
 'stop_reason': None,
 'completed': False,
 'error': 'ValidationException: The provided model identifier is invalid.',
 'time_to_first_token': None,
 'elapsed': None}